In [2]:
import duckdb

DB_PATH = r"C:\Users\Mohit sharma\Desktop\7th sem\new major project\geopulse\datasets\traffic_v1.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)

print("\n=== TABLES ===")
tables = con.execute("SHOW TABLES").fetchdf()
print(tables)

print("\n=== DATABASE SCHEMA ===")

for table in tables["name"]:
    print(f"\n--- {table} ---")
    schema = con.execute(f"DESCRIBE {table}").fetchdf()
    print(schema.to_string(index=False))

con.close()


=== TABLES ===
      name
0  traffic

=== DATABASE SCHEMA ===

--- traffic ---
       column_name              column_type null  key default extra
        segment_id                   BIGINT  YES None    None  None
              hour TIMESTAMP WITH TIME ZONE  YES None    None  None
         timestamp TIMESTAMP WITH TIME ZONE  YES None    None  None
      currentSpeed                   DOUBLE  YES None    None  None
     freeFlowSpeed                   DOUBLE  YES None    None  None
 currentTravelTime                   DOUBLE  YES None    None  None
freeFlowTravelTime                   DOUBLE  YES None    None  None
        confidence                   DOUBLE  YES None    None  None
       is_observed                  BOOLEAN  YES None    None  None
        is_missing                  BOOLEAN  YES None    None  None
       roadClosure                  BOOLEAN  YES None    None  None
             osmid                  VARCHAR  YES None    None  None
         road_name                  

In [3]:
import duckdb
import pandas as pd

DB_PATH = r"C:\Users\Mohit sharma\Desktop\7th sem\new major project\geopulse\datasets\traffic_v1.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)

query = """
SELECT DISTINCT
    segment_id,
    osmid,
    road_name,
    road_type,
    length_m,
    lat,
    lon
FROM traffic
WHERE osmid IS NOT NULL
ORDER BY segment_id
"""

df = con.execute(query).fetchdf()

print("\n========== OSM METADATA ==========")
print(df.head(20).to_string(index=False))

print("\n========== SUMMARY ==========")
print("Rows:", len(df))
print("Unique segments:", df["segment_id"].nunique())
print("Unique OSM IDs:", df["osmid"].nunique())
print("Missing OSM IDs:", df["osmid"].isna().sum())
print("Missing coordinates:", df[["lat", "lon"]].isna().any(axis=1).sum())

print("\n========== ROAD TYPES ==========")
print(df["road_type"].value_counts(dropna=False))

print("\n========== SAMPLE OSM IDs ==========")
print(df["osmid"].head(30).tolist())

output_path = "osm_segment_metadata.csv"
df.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")

con.close()


========== OSM METADATA ==========
 segment_id                                                      osmid  road_name road_type    length_m       lat       lon
          0                        [1209033472, 280421140, 1317533102]        0.0     trunk 3016.997938 20.283950 85.806267
          1                         [44948707, 1317533101, 1317533103]        1.0     trunk 2971.283591 20.283755 85.806173
          2                        [279958946, 1317535730, 1209009343]        2.0     trunk 2001.051040 20.296488 85.838203
          3              [26755549, 1317535729, 1209009372, 218526245]        3.0     trunk 1989.572233 20.296617 85.838137
          4                                                  873398460        4.0     trunk 1656.799982 20.262789 85.868317
          5                           [742084225, 742084226, 44947113]        5.0     trunk 1588.023180 20.347754 85.886387
          6 [44948644, 1203900681, 1194664843, 1194664844, 1317520819]        6.0     trunk 1505